# Sirkelbevegelse med sympy og simulering

Denne notatboken kjører Python rett i nettleseren. Kjør cellene i rekkefølge med **Shift + Enter**. Første gang tar det litt tid før Python er klar.

Oppgavene står på nettsiden: [Sirkelbevegelse med sympy og simulering](https://fysikklab.no/programmering/python-sirkelbevegelse.html).

> **Lagring:** Endringene dine lagres i denne nettleseren på denne maskinen. Vil du ta vare på arbeidet eller levere det, velger du *File → Download*.

## Del 1: Sympy i stedet for CAS

### Kjeglependelen

Kreftene på kula gir $S\cos\theta = mg$ og $S\sin\theta = \dfrac{mv^2}{r}$, med $r = L\sin\theta$. Vi lar sympy løse for $v^2$ i stedet for $v$. Da blir likningssettet lineært, og vi slipper negative røtter.

In [ ]:
import sympy as sp

m, g, L, theta, S, v2 = sp.symbols("m g L theta S v^2", positive=True)
r = L*sp.sin(theta)

lik1 = sp.Eq(S*sp.cos(theta), m*g)       # loddrett
lik2 = sp.Eq(S*sp.sin(theta), m*v2/r)    # horisontalt, mot sentrum

losning = sp.solve([lik1, lik2], [S, v2], dict=True)[0]
print("S =", losning[S])
print("v² =", losning[v2])

T = sp.sqrt(sp.simplify(4*sp.pi**2*r**2/losning[v2]))
print("T =", T)

verdier = {L: 0.800, theta: sp.rad(30), g: 9.81}
print("v =", sp.sqrt(losning[v2]).subs(verdier).evalf(3), "m/s")
print("T =", T.subs(verdier).evalf(3), "s")

# Vis svarene pent
display(sp.Eq(S, losning[S]), sp.Eq(v2, losning[v2]), sp.Eq(sp.Symbol("T"), T))

### Dosert sving med friksjon

Ved største fart peker friksjonen $\mu N$ nedover langs veibanen:

$$N\cos\theta - \mu N\sin\theta = mg \qquad N\sin\theta + \mu N\cos\theta = \frac{mv^2}{r}$$

Ved minste fart peker friksjonen oppover. Det tilsvarer å bytte $\mu$ med $-\mu$.

In [ ]:
import sympy as sp

m, g, r, theta, mu, N, v2 = sp.symbols("m g r theta mu N v^2", positive=True)

lik1 = sp.Eq(N*sp.cos(theta) - mu*N*sp.sin(theta), m*g)
lik2 = sp.Eq(N*sp.sin(theta) + mu*N*sp.cos(theta), m*v2/r)

losning = sp.solve([lik1, lik2], [N, v2], dict=True)[0]
v_maks_kvadrat = sp.factor(losning[v2])
print("v_maks² =", v_maks_kvadrat)

bok = r*g*(sp.tan(theta) + mu)/(1 - mu*sp.tan(theta))
print("Differanse mot formelen i boka:", sp.simplify(v_maks_kvadrat - bok))

v_maks = sp.sqrt(v_maks_kvadrat)
v_min = v_maks.subs(mu, -mu)
tall = {r: 120, theta: sp.rad(17.8), g: 9.81, mu: 0.10}
print(f"v_min = {float(v_min.subs(tall))*3.6:.0f} km/h, v_maks = {float(v_maks.subs(tall))*3.6:.0f} km/h")

display(sp.Eq(sp.Symbol("v_maks")**2, v_maks_kvadrat))

## Del 2: Simuleringer

Når kraftsummen avhenger av farten eller vinkelen, bruker vi Eulers metode med vinkelen som posisjon: $\Delta\theta = \dfrac{v}{r}\Delta t$.

### Snøkuppelen

Tiril glir fra toppen av en glatt snøkuppel med radius 12,0 m. Hun har kontakt så lenge $N = mg\cos\theta - \dfrac{mv^2}{R} > 0$. Når hun mister kontakten, fortsetter hun som et skrått kast.

In [ ]:
import numpy as np

g = 9.81
R = 12.0
mu = 0.0       # friksjonstallet, sett til 0.10 i oppgave 6
v0 = 0.10      # startfart i m/s, sett til 2.0 i oppgave 6
dt = 0.0001

# På kuppelen: vinkelen theta fra toppen er posisjonen
theta = 0.0
v = v0
while True:
    N_per_m = g*np.cos(theta) - v**2/R
    if N_per_m < 0:
        break
    if v <= 0:
        print("Tiril stopper på kuppelen.")
        break
    a = g*np.sin(theta) - mu*N_per_m
    theta = theta + v/R*dt
    v = v + a*dt

print(f"Mister kontakten ved {np.degrees(theta):.1f} grader (teori uten friksjon: {np.degrees(np.arccos(2/3)):.1f} grader)")
print(f"Høyde: {R*np.cos(theta):.2f} m, fart: {v:.2f} m/s")

# Skrått kast med origo i sentrum av kuppelen
x, y = R*np.sin(theta), R*np.cos(theta)
vx, vy = v*np.cos(theta), -v*np.sin(theta)
while y > 0:
    vy = vy - g*dt
    x = x + vx*dt
    y = y + vy*dt

print(f"Lander {x:.1f} m fra sentrum, {x - R:.1f} m utenfor kanten")

### Loop med friksjon

En vogn kjører inn i en loop med radius 10,0 m. Friksjonen er $\mu N$ med $\mu = 0{,}080$. Funksjonen `simuler(v0, mu)` returnerer listene og om vogna klarte loopen.

In [ ]:
import numpy as np

g = 9.81      # m/s²
r = 10.0      # m
mu = 0.080
dt = 0.0001   # s

def simuler(v0, mu=mu):
    phi = 0.0
    v = v0
    phi_liste, v_liste, N_liste = [phi], [v], [v**2/r + g]
    while phi < 2*np.pi:
        N_per_m = v**2/r + g*np.cos(phi)
        if N_per_m < 0 or v <= 0:          # mister kontakten eller stopper
            return phi_liste, v_liste, N_liste, False
        a = -g*np.sin(phi) - mu*N_per_m
        phi = phi + v/r*dt
        v = v + a*dt
        phi_liste.append(phi)
        v_liste.append(v)
        N_liste.append(v**2/r + g*np.cos(phi))
    return phi_liste, v_liste, N_liste, True

phi, v, N, klarte = simuler(25)
if klarte:
    print(f"Vogna klarer loopen og har {v[-1]:.1f} m/s når den er tilbake i bunnen")
else:
    print(f"Vogna mister kontakten ved {np.degrees(phi[-1]):.0f} grader med farten {v[-1]:.1f} m/s")

v0 = 20.0
while not simuler(v0)[3]:
    v0 = v0 + 0.1
print(f"Minste fart i bunnen: {v0:.1f} m/s (uten friksjon: {np.sqrt(5*g*r):.1f} m/s)")

### Normalkraften gjennom loopen

Grafen viser $N/(mg)$ gjennom loopen for tre startfarter. Når kurven når null, mister vogna kontakten med banen.

In [ ]:
import matplotlib.pyplot as plt

for v0 in [24, 25.8, 28]:
    phi, v, N, klarte = simuler(v0)
    plt.plot(np.degrees(phi), np.array(N)/g, label=f"$v_0$ = {v0} m/s")

plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("vinkel fra bunnen / grader")
plt.ylabel("N / (mg)")
plt.legend()
plt.grid(True)
plt.show()

## Din egen kode

Bruk cellen under til oppgavene. Du kan lage flere celler med **+** i verktøylinja.